In [ ]:
!pip install --quiet flax optax einops pillow transformers matplotlib tqdm scikit-image


In [ ]:
!mkdir -p itaclip/{modules,utils,ckpts}
!touch itaclip/__init__.py itaclip/modules/__init__.py itaclip/utils/__init__.py


In [ ]:
%%writefile itaclip/config.py
from dataclasses import dataclass

@dataclass
class Config:
    image_size: int = 224
    patch_size: int = 16
    vit_dim: int = 256
    vit_depth: int = 6
    vit_heads: int = 8
    text_dim: int = 256
    text_vocab_size: int = 49408
    text_max_len: int = 32
    text_heads: int = 8
    text_depth: int = 4
    dropout: float = 0.1


In [ ]:
%%writefile itaclip/modules/vision_encoder.py
from flax import linen as nn
import jax.numpy as jnp
from einops import rearrange
from typing import Sequence
import jax
jax.config.update("jax_debug_nans", False)


class PatchEmbed(nn.Module):
    patch_size: int
    emb_dim: int
    @nn.compact
    def __call__(self, x):
        x = nn.Conv(self.emb_dim, (self.patch_size, self.patch_size),
                    strides=(self.patch_size, self.patch_size))(x)
        return x

class SelfSelfAttention(nn.Module):
    emb_dim: int
    heads: int
    mode: Sequence[str] = ("qk",)

    @nn.compact
    def __call__(self, x):
        d = x.shape[-1]
        q = nn.Dense(d)(x)
        k = nn.Dense(d)(x)
        v = nn.Dense(d)(x)
        q = q.reshape(*q.shape[:-1], self.heads, d // self.heads)
        k = k.reshape(*k.shape[:-1], self.heads, d // self.heads)
        v = v.reshape(*v.shape[:-1], self.heads, d // self.heads)

        logits = 0
        if "qk" in self.mode:
            logits += jnp.einsum("bnhd,bmhd->bhnm", q, k) / jnp.sqrt(d // self.heads)
        if "qq" in self.mode:
            logits += jnp.einsum("bnhd,bmhd->bhnm", q, q) / jnp.sqrt(d // self.heads)
        if "kk" in self.mode:
            logits += jnp.einsum("bnhd,bmhd->bhnm", k, k) / jnp.sqrt(d // self.heads)

        attn = nn.softmax(logits, axis=-1)
        out = jnp.einsum("bhnm,bmhd->bnhd", attn, v)
        out = out.reshape(x.shape)
        out = nn.Dense(d)(out)
        return out, attn

class TinyViT(nn.Module):
    emb_dim: int
    depth: int
    heads: int
    mlp_dim: int = None
    dropout: float = 0.1
    attn_modes: Sequence[str] = ("qk",)
    remove_ffn_last: bool = False
    return_attentions: bool = False

    @nn.compact
    def __call__(self, x, *, train=False):
        B,H,W,C = x.shape
        seq = rearrange(x, "b h w c -> b (h w) c")
        attn_list = []
        for i in range(self.depth):
            y = nn.LayerNorm()(seq)
            attn_out, attn_map = SelfSelfAttention(self.emb_dim, self.heads, mode=self.attn_modes)(y)
            seq = seq + nn.Dropout(self.dropout)(attn_out, deterministic=not train)

            if not (self.remove_ffn_last and i == self.depth - 1):
                z = nn.LayerNorm()(seq)
                mlp_dim = self.mlp_dim or self.emb_dim * 4
                z = nn.Dense(mlp_dim)(z)
                z = nn.gelu(z)
                z = nn.Dense(self.emb_dim)(z)
                seq = seq + nn.Dropout(self.dropout)(z, deterministic=not train)

            attn_list.append(attn_map)

        out = rearrange(seq, "b (h w) c -> b h w c", h=H, w=W)
        return (out, attn_list) if self.return_attentions else out


In [ ]:
%%writefile itaclip/modules/text_encoder.py
from flax import linen as nn
import jax.numpy as jnp

class TinyTextTransformer(nn.Module):
    vocab_size: int
    max_len: int
    emb_dim: int
    depth: int
    heads: int
    dropout: float = 0.1

    @nn.compact
    def __call__(self, token_ids, *, train=False):
        x = nn.Embed(self.vocab_size, self.emb_dim)(token_ids)
        pos = self.param("pos_emb", nn.initializers.normal(0.02),
                         (1, self.max_len, self.emb_dim))
        x = x + pos[:, :x.shape[1], :]
        for _ in range(self.depth):
            y = nn.LayerNorm()(x)
            y = nn.SelfAttention(num_heads=self.heads, qkv_features=self.emb_dim)(y)
            x = x + nn.Dropout(self.dropout)(y, deterministic=not train)
            z = nn.LayerNorm()(x)
            z = nn.Dense(self.emb_dim * 4)(z)
            z = nn.gelu(z)
            z = nn.Dense(self.emb_dim)(z)
            x = x + nn.Dropout(self.dropout)(z, deterministic=not train)
        return jnp.mean(x, axis=1)

In [ ]:
%%writefile itaclip/modules/fusion_head.py
from flax import linen as nn
import jax.numpy as jnp
from einops import repeat, rearrange

class SimpleFusionHead(nn.Module):
    emb_dim: int

    @nn.compact
    def __call__(self, image_feats, text_feat):
        B,H,W,C = image_feats.shape
        img_p = nn.Dense(self.emb_dim)(image_feats)
        if len(text_feat.shape) == 3:
            text_feat = jnp.mean(text_feat, axis=1)
        txt_p = nn.Dense(self.emb_dim)(text_feat)
        txt_spatial = repeat(txt_p, "b c -> b h w c", h=H, w=W)
        cat = jnp.concatenate([img_p, txt_spatial], axis=-1)
        x = nn.Conv(self.emb_dim, (1,1))(cat)
        x = nn.gelu(x)
        logits = nn.Conv(1, (1,1))(x)
        return rearrange(logits, "b h w 1 -> b h w")


In [ ]:
%%writefile itaclip/inference_itaclip.py
import jax, jax.numpy as jnp, numpy as np
from itaclip.utils.preprocess import preprocess_image, tokenize_text
from itaclip.utils.image_engineering import first_category_augs, second_category_augs, normalize_and_stack
from itaclip.utils.aux_text import generate_aux_texts
from itaclip.modules.text_encoder import TinyTextTransformer

def compute_itaclip_logits(params, model, pil_img, class_name, cfg, lambda_img=0.6):
    imgs1 = first_category_augs(pil_img)
    imgs2 = second_category_augs(pil_img)

    feats1 = []
    for im in imgs1:
        arr = preprocess_image(im, cfg.image_size)[None, ...]
        (out, attns) = model.apply(params, arr,
                                   jnp.zeros((1, cfg.text_max_len), jnp.int32),
                                   train=False)
        feat = np.array(attns[-1].mean(axis=1))
        feats1.append(feat.reshape(-1, feat.shape[-1]))

    Xvisual1 = normalize_and_stack(feats1)
    aux_texts = generate_aux_texts(class_name)
    toks = [tokenize_text(t, cfg.text_max_len)[0] for t in aux_texts]
    toks = jnp.array(toks)

    text_model = TinyTextTransformer(cfg.text_vocab_size, cfg.text_max_len,
                                     cfg.text_dim, cfg.text_depth, cfg.text_heads)
    text_params = params["params"]["text"]  
    text_feats = text_model.apply({"params": text_params}, toks, train=False)
    text_feat = jnp.mean(text_feats, axis=0)

    sims1 = (Xvisual1 @ np.array(text_feat).T)
    patch_h = cfg.image_size // cfg.patch_size
    L1 = sims1.reshape(patch_h, patch_h)

    L2s = []
    for im, kind in zip(imgs2, ["hflip", "vflip"]):
        arr = preprocess_image(im, cfg.image_size)[None, ...]
        (out, attns) = model.apply(params, arr,
                                   jnp.zeros((1, cfg.text_max_len), jnp.int32),
                                   train=False)
        feat = np.array(attns[-1].mean(axis=1))
        sims = (feat.reshape(-1, feat.shape[-1]) @ np.array(text_feat).T).reshape(patch_h, patch_h)
        if kind == "hflip": sims = np.fliplr(sims)
        if kind == "vflip": sims = np.flipud(sims)
        L2s.append(sims)

    L2 = np.mean(np.stack(L2s, axis=0), axis=0)
    L = lambda_img * L1 + (1 - lambda_img) * L2
    L_upsampled = jax.image.resize(
        jnp.array(L[None, ...]),
        (1, cfg.image_size, cfg.image_size),
        method="bilinear"
    )[0]
    return L_upsampled


In [ ]:
%%writefile itaclip/model.py
from flax import linen as nn
from itaclip.config import Config
from itaclip.modules.vision_encoder import PatchEmbed, TinyViT
from itaclip.modules.text_encoder import TinyTextTransformer
from itaclip.modules.fusion_head import SimpleFusionHead

class ITACLIP(nn.Module):
    cfg: Config

    def setup(self):
        self.patch = PatchEmbed(self.cfg.patch_size, self.cfg.vit_dim)
        self.vit = TinyViT(self.cfg.vit_dim, self.cfg.vit_depth, self.cfg.vit_heads,
                           attn_modes=("qq","kk"), remove_ffn_last=True, return_attentions=True)
        self.text = TinyTextTransformer(self.cfg.text_vocab_size, self.cfg.text_max_len,
                                        self.cfg.text_dim, self.cfg.text_depth, self.cfg.text_heads)
        self.project_text = nn.Dense(self.cfg.vit_dim)
        self.fusion = SimpleFusionHead(self.cfg.vit_dim)

    def __call__(self, image, token_ids, *, train=False):
        x = self.patch(image)
        x, attns = self.vit(x, train=train)
        t = self.text(token_ids, train=train)
        t = self.project_text(t)
        return self.fusion(x, t), attns


In [ ]:
%%writefile itaclip/utils/preprocess.py
import numpy as np
import jax.numpy as jnp
from PIL import Image
from transformers import CLIPTokenizerFast

tokenizer = CLIPTokenizerFast.from_pretrained("openai/clip-vit-base-patch32")

def preprocess_image(pil_image, image_size=224):
    img = pil_image.convert("RGB").resize((image_size, image_size))
    arr = np.array(img).astype(np.float32) / 255.0
    return jnp.array(arr)

def tokenize_text(text, max_len=32):
    out = tokenizer(text, padding="max_length", truncation=True,
                    max_length=max_len, return_tensors="np")
    return out["input_ids"]


In [ ]:
%%writefile itaclip/utils/image_engineering.py
from PIL import Image, ImageFilter, ImageOps
import numpy as np

def first_category_augs(pil_img):
    imgs = [
        pil_img,  # original
        pil_img.filter(ImageFilter.GaussianBlur(radius=2)),
        ImageOps.grayscale(pil_img).convert("RGB")
    ]
    return imgs

def second_category_augs(pil_img):
    imgs = [
        pil_img.transpose(Image.FLIP_LEFT_RIGHT),
        pil_img.transpose(Image.FLIP_TOP_BOTTOM)
    ]
    return imgs

def reverse_second_aug(img, kind):
    if kind == "hflip":
        return img.transpose(Image.FLIP_LEFT_RIGHT)
    if kind == "vflip":
        return img.transpose(Image.FLIP_TOP_BOTTOM)
    return img

def normalize_and_stack(feats):
    out = []
    for f in feats:
        norm = np.linalg.norm(f.reshape(-1, f.shape[-1]), ord='fro') + 1e-6
        out.append(f / norm)
    return np.mean(np.stack(out, axis=0), axis=0)


In [ ]:
%%writefile itaclip/utils/aux_text.py
import random
AUX_PATTERNS = [
    "a photo of {}",
    "a detailed image of {}",
    "a picture showing {}",
    "a scene containing {}",
    "an object that looks like {}",
    "a real-world example of {}",
    "a texture similar to {}",
    "the concept of {}"
]

def generate_aux_texts(class_name, n=5):
    """
    Generate n auxiliary text prompts for a given class name.
    Example: "class_43" -> ["a photo of class_43", "a scene containing class_43", ...]
    """
    aux = [p.format(class_name) for p in random.sample(AUX_PATTERNS, min(n, len(AUX_PATTERNS)))]
    aux.insert(0, class_name)  # Always include the original label as the first one
    return aux


In [ ]:
%%writefile itaclip/train_itaclip.py

import os, random, pickle, glob
from PIL import Image
import numpy as np
from tqdm import trange
import jax, jax.numpy as jnp
from flax.training import train_state
import optax
from itaclip.model import ITACLIP
from itaclip.config import Config
from itaclip.utils.preprocess import preprocess_image, tokenize_text

cfg = Config()
ADE_PATH = "/kaggle/input/ade20k-dataset/ADEChallengeData2016"
os.makedirs("itaclip/ckpts", exist_ok=True)

def load_ade20k_paths(split="training"):
    img_dir = os.path.join(ADE_PATH, "images", split)
    ann_dir = os.path.join(ADE_PATH, "annotations", split)
    imgs = sorted(glob.glob(f"{img_dir}/*.jpg"))
    anns = sorted(glob.glob(f"{ann_dir}/*.png"))
    return list(zip(imgs, anns))

def sample_class_and_mask(seg):
    unique = np.unique(seg)
    unique = unique[unique > 0]
    if len(unique) == 0:
        return 0, (seg == 0).astype(np.float32)
    c = int(random.choice(unique))
    return c, (seg == c).astype(np.float32)

def ade20k_generator(split="training"):
    pairs = load_ade20k_paths(split)
    while True:
        img_p, ann_p = random.choice(pairs)
        img = Image.open(img_p)
        ann = np.array(Image.open(ann_p))
        pil = img.resize((cfg.image_size, cfg.image_size))
        ann_r = np.array(Image.fromarray(ann).resize((cfg.image_size, cfg.image_size), Image.NEAREST))
        cls, mask = sample_class_and_mask(ann_r)
        text = f"class_{cls}"
        img_arr = preprocess_image(pil, cfg.image_size)
        tok = tokenize_text(text, cfg.text_max_len)[0]
        yield {"image": img_arr, "token_ids": tok, "mask": mask.astype(np.float32)}

def bce_loss(logits, targets):
    return jnp.mean(optax.sigmoid_binary_cross_entropy(logits, targets))

class TrainState(train_state.TrainState):
    pass

def create_state(rng, lr=3e-4):
    model = ITACLIP(cfg)
    di = jnp.zeros((1, cfg.image_size, cfg.image_size, 3))
    dt = jnp.zeros((1, cfg.text_max_len), jnp.int32)
    params = model.init(rng, di, dt)
    tx = optax.adamw(lr)
    return TrainState.create(apply_fn=model.apply, params=params, tx=tx), model

@jax.jit
def train_step(state, batch, rng):
    def loss_fn(params):
        (logits, _) = state.apply_fn(params, batch["image"], batch["token_ids"],
                                     train=True, rngs={"dropout": rng})
        logits_up = jax.image.resize(logits, (logits.shape[0], cfg.image_size, cfg.image_size), method="bilinear")
        loss = bce_loss(logits_up, batch["mask"])
        return loss, logits_up
    (loss, _), grads = jax.value_and_grad(loss_fn, has_aux=True)(state.params)
    state = state.apply_gradients(grads=grads)
    return state, loss

def compute_iou(pred, gt, th=0.5):
    p = (pred > th).astype(np.uint8)
    g = (gt > 0.5).astype(np.uint8)
    inter = (p & g).sum()
    union = (p | g).sum()
    return float(inter / (union + 1e-6))

def train_ade20k(epochs=2, steps_per_epoch=200, lr=3e-4):
    rng = jax.random.PRNGKey(0)
    state, model = create_state(rng, lr)
    train_gen = ade20k_generator("training")
    val_gen = ade20k_generator("validation")

    for ep in range(1, epochs + 1):
        rng, ep_rng = jax.random.split(rng)
        for step in trange(steps_per_epoch, desc=f"Epoch {ep}/{epochs}"):
            sample = next(train_gen)
            batch = {
                "image": jnp.expand_dims(sample["image"], 0),
                "token_ids": jnp.expand_dims(sample["token_ids"], 0),
                "mask": jnp.expand_dims(sample["mask"], 0)
            }
            ep_rng, step_rng = jax.random.split(ep_rng)
            state, loss = train_step(state, batch, step_rng)
            if step % 50 == 0:
                print(f"Step {step} | Loss={loss:.4f}")

        # Validation
        s = next(val_gen)
        val_img = jnp.expand_dims(s["image"], 0)
        val_tok = jnp.expand_dims(s["token_ids"], 0)
        val_mask = s["mask"]
        logits, _ = model.apply(state.params, val_img, val_tok, train=False)
        logits_up = jax.image.resize(logits, (1, cfg.image_size, cfg.image_size), method="bilinear")
        preds = jax.nn.sigmoid(logits_up)
        iou = compute_iou(np.array(preds[0]), val_mask)
        print(f"✅ Epoch {ep} done | Val IoU={iou:.4f}")

        # Save checkpoint
        ckpt_path = f"itaclip/ckpts/params_epoch{ep}.pkl"
        with open(ckpt_path, "wb") as f:
            pickle.dump(state.params, f)
        print(f"💾 Saved checkpoint → {ckpt_path}")

if __name__ == "__main__":
    train_ade20k(epochs=20, steps_per_epoch=200)

In [ ]:
!python -m itaclip.train_itaclip


In [ ]:
import os, random, pickle, jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from itaclip.model import ITACLIP
from itaclip.config import Config
from itaclip.utils.preprocess import preprocess_image, tokenize_text

cfg = Config()
ADE_PATH = "/kaggle/input/ade20k-dataset/ADEChallengeData2016"

# === 1️⃣ Load latest checkpoint ===
ckpts = sorted([f for f in os.listdir("itaclip/ckpts") if f.endswith(".pkl")])
if not ckpts:
    raise FileNotFoundError("❌ No checkpoints found. Please run training first.")
latest = os.path.join("itaclip/ckpts", ckpts[-1])
print(f"📦 Using checkpoint: {latest}")

with open(latest, "rb") as f:
    params = pickle.load(f)

# === 2️⃣ Initialize model ===
model = ITACLIP(cfg)
_ = model.init(jax.random.PRNGKey(0),
               jnp.zeros((1, cfg.image_size, cfg.image_size, 3)),
               jnp.zeros((1, cfg.text_max_len), jnp.int32))

# === 3️⃣ Pick random validation image ===
img_dir = os.path.join(ADE_PATH, "images/validation")
ann_dir = os.path.join(ADE_PATH, "annotations/validation")
img_file = random.choice(sorted(os.listdir(img_dir)))
img_path = os.path.join(img_dir, img_file)
ann_path = os.path.join(ann_dir, img_file.replace(".jpg", ".png"))
img = Image.open(img_path)
ann = np.array(Image.open(ann_path))

# === 4️⃣ Pick a random class ===
cls = int(random.choice(np.unique(ann)))
prompt = f"class_{cls}"
print(f"🖼️ {img_file} | 🎯 {prompt}")

# === 5️⃣ Preprocess and predict ===
pil = img.resize((cfg.image_size, cfg.image_size))
img_jax = preprocess_image(pil, cfg.image_size)
txt = tokenize_text(prompt, cfg.text_max_len)
txt = jnp.array(txt).astype(jnp.int32)

img_batch = jnp.expand_dims(img_jax, 0)
txt_batch = jnp.expand_dims(txt, 0)
logits, _ = model.apply(params, img_batch, txt_batch, train=False)

# Upsample logits to match image size
logits_up = jax.image.resize(logits, (1, cfg.image_size, cfg.image_size), method="bilinear")
mask_pred = np.array(jax.nn.sigmoid(logits_up[0]))
mask_gt = np.array(Image.fromarray(ann).resize((cfg.image_size, cfg.image_size), Image.NEAREST)) == cls

# === 6️⃣ Overlay visualization ===
base_img = np.array(pil)
color_mask = plt.cm.jet(mask_pred.squeeze())[:, :, :3]
overlay = (0.6 * base_img + 0.4 * (color_mask * 255)).astype(np.uint8)

# === 7️⃣ Display results ===
plt.figure(figsize=(14, 5))
plt.subplot(1, 3, 1)
plt.imshow(base_img)
plt.axis("off")
plt.title("Input Image")

plt.subplot(1, 3, 2)
plt.imshow(mask_pred, cmap="gray")
plt.axis("off")
plt.title("Predicted Mask")

plt.subplot(1, 3, 3)
plt.imshow(overlay)
plt.axis("off")
plt.title("Overlay (Blended)")

plt.tight_layout()
plt.show()


In [ ]:
import os, random, pickle, jax, jax.numpy as jnp, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from itaclip.model import ITACLIP
from itaclip.config import Config
from itaclip.utils.preprocess import preprocess_image, tokenize_text

cfg = Config()
ADE_PATH = "/kaggle/input/ade20k-dataset/ADEChallengeData2016"

# === 1️⃣ Load latest checkpoint ===
ckpts = sorted([f for f in os.listdir("itaclip/ckpts") if f.endswith(".pkl")])
if not ckpts:
    raise FileNotFoundError("❌ No checkpoints found. Please run training first.")
latest = os.path.join("itaclip/ckpts", ckpts[-1])
print(f"📦 Using checkpoint: {latest}")

with open(latest, "rb") as f:
    params = pickle.load(f)

# === 2️⃣ Initialize model ===
model = ITACLIP(cfg)
_ = model.init(jax.random.PRNGKey(0),
               jnp.zeros((1, cfg.image_size, cfg.image_size, 3)),
               jnp.zeros((1, cfg.text_max_len), jnp.int32))

# === 3️⃣ Pick random validation image ===
img_dir = os.path.join(ADE_PATH, "images/validation")
ann_dir = os.path.join(ADE_PATH, "annotations/validation")
img_file = random.choice(sorted(os.listdir(img_dir)))
img_path = os.path.join(img_dir, img_file)
ann_path = os.path.join(ann_dir, img_file.replace(".jpg", ".png"))
img = Image.open(img_path)
ann = np.array(Image.open(ann_path))

# === 4️⃣ Pick a random class ===
cls = int(random.choice(np.unique(ann)))
prompt = f"class_{cls}"
print(f"🖼️ {img_file} | 🎯 {prompt}")

# === 5️⃣ Preprocess and predict ===
pil = img.resize((cfg.image_size, cfg.image_size))
img_jax = preprocess_image(pil, cfg.image_size)
txt = tokenize_text(prompt, cfg.text_max_len)
txt = jnp.array(txt).astype(jnp.int32)

img_batch = jnp.expand_dims(img_jax, 0)
txt_batch = jnp.expand_dims(txt, 0)
logits, _ = model.apply(params, img_batch, txt_batch, train=False)

# Upsample logits
logits_up = jax.image.resize(logits, (1, cfg.image_size, cfg.image_size), method="bilinear")
mask_pred = np.array(jax.nn.sigmoid(logits_up[0]))
mask_gt = np.array(Image.fromarray(ann).resize((cfg.image_size, cfg.image_size), Image.NEAREST)) == cls

# === 6️⃣ Compute IoU ===
def compute_iou(pred, gt, th=0.5):
    p = (pred > th).astype(np.uint8)
    g = gt.astype(np.uint8)
    inter = (p & g).sum()
    union = (p | g).sum()
    return float(inter / (union + 1e-6))

iou = compute_iou(mask_pred, mask_gt)
print(f"📊 IoU for class {cls}: {iou:.4f}")

# === 7️⃣ Overlay visualization ===
base_img = np.array(pil)
color_mask = plt.cm.jet(mask_pred.squeeze())[:, :, :3]
overlay = (0.6 * base_img + 0.4 * (color_mask * 255)).astype(np.uint8)

# === 8️⃣ Display results ===
plt.figure(figsize=(18, 5))
plt.subplot(1, 4, 1)
plt.imshow(base_img)
plt.axis("off")
plt.title("Input Image")

plt.subplot(1, 4, 2)
plt.imshow(mask_gt, cmap="gray")
plt.axis("off")
plt.title(f"Ground Truth (class_{cls})")

plt.subplot(1, 4, 3)
plt.imshow(mask_pred, cmap="gray")
plt.axis("off")
plt.title("Predicted Mask")

plt.subplot(1, 4, 4)
plt.imshow(overlay)
plt.axis("off")
plt.title(f"Overlay (IoU={iou:.3f})")

plt.tight_layout()
plt.show()
